In [1]:
import sys
sys.path.append('../')
import os
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

import torch
import numpy as np
from tqdm import tqdm
from torchvision import datasets, transforms

from Network.ReActNet_18_Qaw import *
from Network.ReActNet_A_Qaw import *
from Network.utils import *

In [2]:
resnet18 = True

In [3]:
Dataset_path = 'F:/Dataset/CIFAR-100/'
Batch_size = 256
Workers = 0
Targetnum = 100

In [4]:
_seed_ = 2023
torch.manual_seed(_seed_)
np.random.seed(_seed_)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [5]:
transform_train = transforms.Compose([
    transforms.Pad(4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
 
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

Train_data = datasets.CIFAR100(root=Dataset_path, train=True, download=False, transform=transform_train)
Test_data = datasets.CIFAR100(root=Dataset_path, train=False, download=False, transform=transform_test)

train_data_loader = torch.utils.data.DataLoader(
    dataset=Train_data,
    batch_size=Batch_size,
    shuffle=True,
    num_workers=Workers, 
    pin_memory=True,
    drop_last=True
)

test_data_loader = torch.utils.data.DataLoader(
    dataset=Test_data,
    batch_size=Batch_size,
    shuffle=False,
    num_workers=Workers, 
    pin_memory=True,
    drop_last=False
)

In [6]:
net = ResNet18(num_classes=Targetnum, imagenet=False) if resnet18 else Reactnet(num_classes=Targetnum, imagenet=False)
net = torch.nn.DataParallel(net).cuda()
net_model_path = './models/cifar100_reactnet18_step2.h5' if resnet18 else './models/cifar100_reactneta_step2.h5'
net.load_state_dict(torch.load(net_model_path, weights_only=False))

<All keys matched successfully>

In [7]:
Confusion_Matrix = torch.zeros((Targetnum, Targetnum))
net.eval()
with torch.no_grad():
    for img, label in tqdm(test_data_loader):
        img = img.cuda()
        label = label.cuda()
        out_fr = net(img)
        guess = out_fr.argmax(1)
        for j in range(len(label)):
            Confusion_Matrix[label[j],guess[j]] += 1
acc = Confusion_Matrix.diag()
acc = acc.sum()/Confusion_Matrix.sum()
print(f'Confusion_Matrix = \n{Confusion_Matrix}')
print(f'acc = {acc}')

100%|██████████| 40/40 [00:07<00:00,  5.15it/s]

Confusion_Matrix = 
tensor([[87.,  0.,  0.,  ...,  0.,  0.,  0.],
        [ 0., 82.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  1., 64.,  ...,  0.,  3.,  0.],
        ...,
        [ 0.,  0.,  0.,  ..., 67.,  0.,  0.],
        [ 1.,  1.,  3.,  ...,  1., 43.,  1.],
        [ 0.,  0.,  1.,  ...,  0.,  0., 71.]])
acc = 0.691100001335144
